# Prática 2 — Fine-Tuning em CIFAR-10 (Versão Aluno)
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook você irá implementar o conceito de **Fine-Tuning parcial** e compará-lo com a **Feature Extraction**:
1. Entender a diferença teórica de descongelar partes do backbone.
2. Implementar a estratégia de **Discriminative Learning Rates** (taxas de aprendizado discriminativas).
3. Comparar o tempo de processamento e a acurácia de validação obtida.
4. Gerar matrizes de confusão para analisar os erros de cada abordagem.

Complete as seções marcadas com `### SEU CÓDIGO AQUI ###`.

## 0. Imports e Configurações

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import time

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Subset Balanceado

In [ ]:
N_TRAIN = 5000
N_VAL = 1000

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

def extract_balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, N_TRAIN)
val_dataset = extract_balanced_subset(val_full, N_VAL)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

## 2. Funções Gerais de Loops

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, label=''):
    history = {'loss': [], 'acc': []}
    for epoch in range(num_epochs):
        t0 = time.time()
        loss = train_epoch(model, train_loader, criterion, optimizer, device)
        acc, _, _ = evaluate(model, val_loader, device)
        elapsed = time.time() - t0
        history['loss'].append(loss)
        history['acc'].append(acc)
        print(f'[{label}] Época {epoch+1}/{num_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | {elapsed:.1f}s')
    return history

## 3. Experimento A — Feature Extraction (Linha de base)

Repetimos a linha de base de Feature Extraction para comparação direta de tempo e acurácia.

In [ ]:
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in model_fe.parameters():
    p.requires_grad = False

model_fe.fc = nn.Linear(model_fe.fc.in_features, 10)
model_fe = model_fe.to(device)

optimizer_fe = optim.SGD(model_fe.fc.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

print('=== Treinamento: Feature Extraction ===')
history_fe = train_model(model_fe, train_loader, val_loader, criterion, optimizer_fe, num_epochs=5, label='FE')

## 4. Experimento B — Fine-Tuning Parcial com Taxas Discriminativas

In [ ]:
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# ==========================================
# EXERCÍCIO 1: Descongelamento Seletivo
# 1. Congele todos os pesos de model_ft inicialmente
# 2. Descongele APENAS os parâmetros pertencentes a model_ft.layer4 (último bloco residual)
# ==========================================
### SEU CÓDIGO AQUI ###


# Substituir a camada final fc
model_ft.fc = nn.Linear(model_ft.fc.in_features, 10)
model_ft = model_ft.to(device)

# ==========================================
# EXERCÍCIO 2: Taxas de Aprendizado Discriminativas
# Configure o otimizador SGD para usar:
# - lr = 1e-4 para os parâmetros de model_ft.layer4.parameters()
# - lr = 1e-3 para os parâmetros de model_ft.fc.parameters()
# Use momentum=0.9
# ==========================================
### SEU CÓDIGO AQUI ###
optimizer_ft = None

trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Parâmetros Treináveis (Fine-Tuning Parcial): {trainable_ft:,}')

print('\n=== Treinamento: Fine-Tuning Parcial ===')
history_ft = train_model(model_ft, train_loader, val_loader, criterion, optimizer_ft, num_epochs=5, label='FT')

## 5. Análise Comparativa e Plot das Curvas

In [ ]:
epochs = range(1, 6)
plt.figure(figsize=(12, 5))

# Gráfico de Acurácia
plt.subplot(1, 2, 1)
plt.plot(epochs, [a*100 for a in history_fe['acc']], 'o-', color='#1f77b4', label='Feature Extraction')
plt.plot(epochs, [a*100 for a in history_ft['acc']], 's-', color='#ff7f0e', label='Fine-Tuning Parcial')
plt.title('Acurácia de Validação por Época')
plt.xlabel('Época')
plt.ylabel('Acurácia (%)')
plt.grid(True, alpha=0.3)
plt.legend()

# Gráfico de Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, history_fe['loss'], 'o-', color='#1f77b4', label='Feature Extraction')
plt.plot(epochs, history_ft['loss'], 's-', color='#ff7f0e', label='Fine-Tuning Parcial')
plt.title('Loss de Treinamento por Época')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Matriz de Confusão Comparativa

In [ ]:
_, preds_fe, labels_fe = evaluate(model_fe, val_loader, device)
_, preds_ft, labels_ft = evaluate(model_ft, val_loader, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

cm_fe = confusion_matrix(labels_fe, preds_fe)
ConfusionMatrixDisplay(confusion_matrix=cm_fe, display_labels=CLASSES).plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Feature Extraction')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

cm_ft = confusion_matrix(labels_ft, preds_ft)
ConfusionMatrixDisplay(confusion_matrix=cm_ft, display_labels=CLASSES).plot(ax=ax2, cmap='Oranges', colorbar=False)
ax2.set_title('Fine-Tuning Parcial (layer4)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 7. Discussão dos Resultados

1. Como o Fine-Tuning Parcial se compara com a Extração de Features em termos de acurácia final?
2. Explique por que é fundamental usar um learning rate muito menor no bloco `layer4` do que na camada `fc` final.